# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List record sets and their field @ids using mlcroissant metadata
print("Record sets and fields:")
record_sets = metadata.recordSet
for record_set in record_sets:
    print(f"RecordSet @id: {record_set['@id']}")
    print(f"  Name: {record_set.get('name', '[No name]')}")
    print(f"  Description: {record_set.get('description', '[No description]')}")
    # List fields belonging to the record set
    fields = record_set.get('field', [])
    for field in fields:
        # Each 'field' is usually a dict with an @id; in many croissant exports, it's a list of objects
        if isinstance(field, dict) and '@id' in field:
            print(f"    Field @id: {field['@id']}")
            print(f"      Name: {field.get('name', '[No name]')}")
        else:
            print(f"    Field reference: {field}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Choose record set(s) by their @id for extraction
# For this dataset, let's identify the available record sets and use one for further exploration
record_set_ids = [rs['@id'] for rs in metadata.recordSet]
print(f"Available record set @ids: {record_set_ids}")

# Let's pick the main record set for proteomic data. We'll select the first one for demonstration.
selected_record_set_id = record_set_ids[0]

# Load records
records = list(dataset.records(record_set=selected_record_set_id))
df = pd.DataFrame(records)

print(f"Columns in DataFrame:")
print(df.columns.tolist())
print(f"First 5 rows:")
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example: Analyzing protein abundance values
# We'll attempt to find a likely numeric column representing abundance, peptide count, MW, or coverage
import numpy as np

numeric_candidates = []
for col in df.columns:
    # Try to convert to numeric and see how many non-nan values are left
    try:
        series = pd.to_numeric(df[col], errors='coerce')
        if series.notna().sum() > 0:
            numeric_candidates.append(col)
    except Exception:
        continue

print(f"Numeric candidates: {numeric_candidates}")

# Let's select the first numeric candidate for demonstration (e.g., 'abundance' or 'MW', etc.)
numeric_field = numeric_candidates[0] if numeric_candidates else None
if numeric_field:
    print(f"Using field '{numeric_field}' for further analysis.")
    # Ensure numeric dtype
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].quantile(0.75) # Select proteins in top quartile for example
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} (top quartile): {len(filtered_df)} records")
    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a likely categorical field if present (e.g., 'Sample' or 'Accession')
    group_field = None
    for col in df.columns:
        if col.lower() in ['accession', 'sample', 'group']:
            group_field = col
            break

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped mean {numeric_field} by {group_field}:")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Plot distribution of the selected numeric field
import matplotlib.pyplot as plt
import seaborn as sns
if numeric_field:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If grouped analysis is available
    if group_field:
        top_groups = df[group_field].value_counts().index[:5]
        plt.figure(figsize=(10,6))
        for grp in top_groups:
            sns.kdeplot(df[df[group_field]==grp][numeric_field].dropna(), label=grp)
        plt.title(f"{numeric_field} Distribution by Top 5 {group_field}")
        plt.xlabel(numeric_field)
        plt.legend()
        plt.show()
else:
    print("No numeric field for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded the FAIR^2 dataset describing mass spectrometry analysis of extracellular vesicles from human mast cells using the `mlcroissant` library. We:
- Discovered record sets and inspected the available fields via Croissant schema `@id` references.
- Extracted records into Pandas DataFrame and identified numeric fields for investigation.
- Filtered, grouped, normalized, and visualized protein abundance (or other key numeric measurements).

This approach can be easily extended to more advanced omics analytics or integrated in computational pipelines for proteomics or biomarker discovery!